# Audited validation of `hamiltonian_djt_jqt.py`

This notebook performs a **regression-equivalence validation**: it checks that
`hamiltonian_djt_jqt.py` reproduces the numerical behavior of the original
QuTiP implementation in `hamiltonian_DJT(1).py` when the Cartesian hyperfine
tensors are set to the exact DJT tensor representation of `Aperp`, `Apar`,
`A1`, and `A2`.

It validates:

1. the scalar-to-tensor hyperfine mapping at ground, excited, and synthetic
   parameter values;
2. each isolated Hamiltonian component, including the small IOC and asymmetric-
   Ham terms, so those terms are not hidden by the much larger spin-orbit term;
3. `J2`, all three dipole operators, `Href`, and complete Hamiltonian matrices;
4. every public return value of `solve_hamiltonian`, including the nested
   eigenstate objects and returned dipole operators;
5. eigenvalues, adjacent energy gaps, alignments, and phase-invariant
   eigenstate projectors for both sweep and scalar-field inputs;
6. `calculate_cyclicity`, `PLE_transitions`,
   `calculate_cyclicity_spinflip`, `_folded_cyclicity_weight`, and
   `PLE_spectrum`;
7. bare, cyclicity-weighted, and scalar-input spectra on frequency samples that
   include the actual transition centers.

Every approximate-equivalence comparison uses `jax.numpy.allclose` with its
standard tolerances, `rtol=1e-5` and `atol=1e-8`. The candidate is passed as the
first argument and the QuTiP reference as the second, so the relative tolerance
is scaled by the reference value. Shape, sequence-length, uniqueness, and
finiteness requirements are checked separately and exactly.

A successful run ends with `ALL AUDITED VALIDATION CHECKS PASSED`.

> Scope: passing this notebook establishes agreement with the original model at
> the tested points. It is not, by itself, an independent proof that the shared
> physical model is correct for every parameter value.

## Required files

Place these files beside the notebook:

- `hamiltonian_DJT(1).py`
- `parameters_DJT(1).py`
- `hamiltonian_djt_jqt.py`

The Python environment must provide `jax`, `jaxquantum`, `numpy`, and `qutip`.

In [1]:
from __future__ import annotations

from pathlib import Path
import importlib.util
import sys

from jax import config

# Enable x64 before importing jax.numpy so THz and sub-MHz terms coexist safely.
config.update("jax_enable_x64", True)

import jax
import jax.numpy as jnp
import numpy as np
import qutip
import jaxquantum

print("JAX version:", getattr(jax, "__version__", "unknown"))
print("JAXQuantum version:", getattr(jaxquantum, "__version__", "unknown"))
print("QuTiP version:", getattr(qutip, "__version__", "unknown"))
print("JAX x64 enabled:", jax.config.read("jax_enable_x64"))

assert jax.config.read("jax_enable_x64"), "JAX x64 mode must be enabled."

JAX version: 0.10.1
JAXQuantum version: 0.3.0
QuTiP version: 5.2.0
JAX x64 enabled: True


## Load the original and new modules

The original filenames contain parentheses, so the notebook loads them with
`importlib`. The parameter file is registered as `parameters_DJT` before either
Hamiltonian module is imported, preserving the imports used by both files.

In [2]:
REQUIRED_FILENAMES = (
    "hamiltonian_DJT(1).py",
    "parameters_DJT(1).py",
    "hamiltonian_djt_jqt.py",
)


def locate_workspace() -> Path:
    """Find a directory containing all validation inputs."""
    candidates = (Path.cwd() / "git" / "qcontrol" / "src" / "qcontrol" / "snv120" / "fromchat", Path("/mnt/data"))
    for candidate in candidates:
        if all((candidate / name).exists() for name in REQUIRED_FILENAMES):
            return candidate.resolve()

    missing_report = {
        str(candidate): [
            name
            for name in REQUIRED_FILENAMES
            if not (candidate / name).exists()
        ]
        for candidate in candidates
    }
    raise FileNotFoundError(
        "Could not find all validation files. Missing by directory: "
        f"{missing_report}"
    )


def load_module(module_name: str, path: Path):
    """Load one Python source file under a chosen module name."""
    sys.modules.pop(module_name, None)
    specification = importlib.util.spec_from_file_location(module_name, path)
    if specification is None or specification.loader is None:
        raise ImportError(f"Could not create an import specification for {path}.")

    module = importlib.util.module_from_spec(specification)
    sys.modules[module_name] = module
    specification.loader.exec_module(module)
    return module


ROOT = locate_workspace()
print("Validation directory:", ROOT)

# Both Hamiltonian modules import this exact module name.
params = load_module("parameters_DJT", ROOT / "parameters_DJT(1).py")
original = load_module(
    "hamiltonian_DJT_original",
    ROOT / "hamiltonian_DJT(1).py",
)
new = load_module(
    "hamiltonian_djt_jqt",
    ROOT / "hamiltonian_djt_jqt.py",
)

print("Loaded original:", ROOT / "hamiltonian_DJT(1).py")
print("Loaded new:", ROOT / "hamiltonian_djt_jqt.py")

Validation directory: /orcd/home/002/floresh2/file_system/git/qcontrol/src/qcontrol/snv120/fromchat
Loaded original: /orcd/home/002/floresh2/file_system/git/qcontrol/src/qcontrol/snv120/fromchat/hamiltonian_DJT(1).py
Loaded new: /orcd/home/002/floresh2/file_system/git/qcontrol/src/qcontrol/snv120/fromchat/hamiltonian_djt_jqt.py


## Comparison and structure helpers

Eigenvectors are compared through rank-one projectors `|psi><psi|`, which are
invariant to arbitrary global eigensolver phases. Nested state objects returned
by `solve_hamiltonian` are independently converted back to column matrices and
checked as well.

In [3]:
RTOL = 1e-5
ATOL = 1e-8
numeric_results: list[dict[str, object]] = []
structure_results: list[dict[str, object]] = []


def record_structure(name: str, passed: bool, detail: str = "") -> bool:
    """Record an exact, non-numerical structure requirement."""
    passed = bool(passed)
    structure_results.append(
        {
            "name": name,
            "passed": passed,
            "detail": detail,
        }
    )
    status = "PASS" if passed else "FAIL"
    suffix = f" -- {detail}" if detail else ""
    print(f"{status:4s}  {name}{suffix}")
    return passed


def require_length(name: str, value, expected_length: int) -> None:
    """Require an exact sequence length and stop before unsafe indexing."""
    actual_length = len(value)
    passed = record_structure(
        name,
        actual_length == expected_length,
        f"expected {expected_length}, received {actual_length}",
    )
    if not passed:
        raise AssertionError(
            f"{name}: expected length {expected_length}, "
            f"received {actual_length}."
        )


def qutip_dense(operator) -> jnp.ndarray:
    """Return one QuTiP operator as a dense JAX array."""
    return jnp.asarray(operator.full())


def jaxquantum_dense(operator) -> jnp.ndarray:
    """Return one JAXQuantum operator as a dense JAX array."""
    return jnp.asarray(operator.to_dense().data)


def state_projectors(eigenvectors) -> jnp.ndarray:
    """Build phase-invariant projectors from batched column eigenvectors."""
    eigenvectors = jnp.asarray(eigenvectors)
    if eigenvectors.ndim != 3:
        raise ValueError(
            "eigenvectors must have shape "
            "(num_fields, dimension, num_states)."
        )
    return jnp.einsum(
        "bik,bjk->bkij",
        eigenvectors,
        jnp.conj(eigenvectors),
    )


def qutip_state_columns(state_batches) -> jnp.ndarray:
    """Convert nested QuTiP states to batched column eigenvector matrices."""
    matrices = []
    for state_batch in state_batches:
        columns = [
            jnp.ravel(jnp.asarray(state.full()))
            for state in state_batch
        ]
        matrices.append(jnp.stack(columns, axis=1))
    return jnp.stack(matrices, axis=0)


def jaxquantum_state_columns(state_batches) -> jnp.ndarray:
    """Convert nested JAXQuantum states to batched column matrices."""
    matrices = []
    for state_batch in state_batches:
        columns = [
            jnp.ravel(jnp.asarray(state.to_dense().data))
            for state in state_batch
        ]
        matrices.append(jnp.stack(columns, axis=1))
    return jnp.stack(matrices, axis=0)


def record_allclose(name: str, expected, actual) -> bool:
    """Compare finite arrays with reference-scaled default `jnp.allclose`."""
    expected = jnp.asarray(expected)
    actual = jnp.asarray(actual)

    same_shape = expected.shape == actual.shape
    expected_finite = bool(jnp.all(jnp.isfinite(expected)))
    actual_finite = bool(jnp.all(jnp.isfinite(actual)))

    if not same_shape:
        passed = False
        max_abs = float("inf")
        max_tolerance_ratio = float("inf")
    elif not expected_finite or not actual_finite:
        passed = False
        max_abs = float("inf")
        max_tolerance_ratio = float("inf")
    else:
        # jnp.allclose(a, b) scales rtol by |b|. Put the QuTiP reference second.
        passed = bool(
            jnp.allclose(actual, expected, rtol=RTOL, atol=ATOL)
        )
        if expected.size == 0:
            max_abs = 0.0
            max_tolerance_ratio = 0.0
        else:
            absolute_difference = jnp.abs(actual - expected)
            tolerance = ATOL + RTOL * jnp.abs(expected)
            max_abs = float(jnp.max(absolute_difference))
            max_tolerance_ratio = float(
                jnp.max(absolute_difference / tolerance)
            )

    numeric_results.append(
        {
            "name": name,
            "passed": passed,
            "max_abs_difference": max_abs,
            "max_tolerance_ratio": max_tolerance_ratio,
            "expected_shape": tuple(expected.shape),
            "actual_shape": tuple(actual.shape),
            "expected_finite": expected_finite,
            "actual_finite": actual_finite,
        }
    )
    status = "PASS" if passed else "FAIL"
    print(
        f"{status:4s}  {name:64s}  "
        f"max |Δ| = {max_abs:.6e}  "
        f"max tolerance ratio = {max_tolerance_ratio:.3e}"
    )
    return passed


def require_nonzero_component(name: str, component) -> None:
    """Ensure an isolated test component is not accidentally the zero matrix."""
    component = jnp.asarray(component)
    nonzero = not bool(
        jnp.allclose(
            component,
            jnp.zeros_like(component),
            rtol=RTOL,
            atol=ATOL,
        )
    )
    passed = record_structure(
        f"{name}: reference component is nonzero",
        nonzero,
    )
    if not passed:
        raise AssertionError(f"{name} produced a zero reference component.")


def independent_djt_tensors(Aperp, Apar, A1, A2):
    """Construct the DJT tensor mapping independently of the new module."""
    dtype = jnp.result_type(Aperp, Apar, A1, A2, jnp.complex128)
    A = jnp.array(
        [
            [Aperp, 0.0, 0.0],
            [0.0, Aperp, 0.0],
            [0.0, 0.0, 2.0 * Apar],
        ],
        dtype=dtype,
    )
    Ax = jnp.array(
        [
            [-0.5 * A2, 0.0, A1],
            [0.0, 0.5 * A2, 0.0],
            [A1, 0.0, 0.0],
        ],
        dtype=dtype,
    )
    Ay = jnp.array(
        [
            [0.0, -0.5 * A2, 0.0],
            [-0.5 * A2, 0.0, -A1],
            [0.0, -A1, 0.0],
        ],
        dtype=dtype,
    )
    return A, Ax, Ay


def manifold_scalars(manifold: str) -> dict[str, float]:
    """Return original DJT scalar hyperfine values for one manifold."""
    if manifold not in {"ground", "excited"}:
        raise ValueError(f"Unsupported manifold: {manifold!r}.")
    suffix = "gnd" if manifold == "ground" else "exc"
    return {
        "Aperp": getattr(params, f"Aperp_{suffix}"),
        "Apar": getattr(params, f"Apar_{suffix}"),
        "A1": getattr(params, f"A1_{suffix}"),
        "A2": getattr(params, f"A2_{suffix}"),
    }


def manifold_tensors(manifold: str):
    """Return the independent tensor representation for one manifold."""
    return independent_djt_tensors(**manifold_scalars(manifold))

## 1. Validate the scalar-to-tensor mapping

The mapping is checked at both physical parameter sets and at a third synthetic
set with distinct signs and magnitudes. Exact tuple length is required before
using `zip(..., strict=True)`, preventing a missing or extra tensor from being
silently ignored.

In [4]:
mapping_cases = (
    ("ground defaults", manifold_scalars("ground")),
    ("excited defaults", manifold_scalars("excited")),
    (
        "synthetic mixed signs",
        {
            "Aperp": -0.371,
            "Apar": 0.219,
            "A1": -0.013,
            "A2": 0.027,
        },
    ),
)

for case_name, scalar_values in mapping_cases:
    expected_tensors = independent_djt_tensors(**scalar_values)
    actual_tensors = new.djt_hyperfine_tensors(**scalar_values)

    require_length(
        f"{case_name}: djt_hyperfine_tensors return length",
        actual_tensors,
        3,
    )

    for tensor_name, expected, actual in zip(
        ("A", "Ax", "Ay"),
        expected_tensors,
        actual_tensors,
        strict=True,
    ):
        record_allclose(
            f"{case_name}: independent mapping for {tensor_name}",
            expected,
            actual,
        )

E0727 11:07:00.891393 1722535 cuda_executor.cc:1273] [0] Failed to allocate device memory: INTERNAL: [0] Failed to allocate 33.29GiB (35750805504 bytes) of device memory: : CUDA_ERROR_OUT_OF_MEMORY: out of memory
E0727 11:07:00.891686 1722535 cuda_executor.cc:1273] [0] Failed to allocate device memory: INTERNAL: [0] Failed to allocate 29.97GiB (32175724544 bytes) of device memory: : CUDA_ERROR_OUT_OF_MEMORY: out of memory
E0727 11:07:00.891945 1722535 cuda_executor.cc:1273] [0] Failed to allocate device memory: INTERNAL: [0] Failed to allocate 26.97GiB (28958150656 bytes) of device memory: : CUDA_ERROR_OUT_OF_MEMORY: out of memory
E0727 11:07:00.892210 1722535 cuda_executor.cc:1273] [0] Failed to allocate device memory: INTERNAL: [0] Failed to allocate 24.27GiB (26062333952 bytes) of device memory: : CUDA_ERROR_OUT_OF_MEMORY: out of memory
E0727 11:07:00.892461 1722535 cuda_executor.cc:1273] [0] Failed to allocate device memory: INTERNAL: [0] Failed to allocate 21.84GiB (23456100352 by

PASS  ground defaults: djt_hyperfine_tensors return length -- expected 3, received 3
PASS  ground defaults: independent mapping for A                        max |Δ| = 0.000000e+00  max tolerance ratio = 0.000e+00
PASS  ground defaults: independent mapping for Ax                       max |Δ| = 0.000000e+00  max tolerance ratio = 0.000e+00
PASS  ground defaults: independent mapping for Ay                       max |Δ| = 0.000000e+00  max tolerance ratio = 0.000e+00
PASS  excited defaults: djt_hyperfine_tensors return length -- expected 3, received 3
PASS  excited defaults: independent mapping for A                       max |Δ| = 0.000000e+00  max tolerance ratio = 0.000e+00
PASS  excited defaults: independent mapping for Ax                      max |Δ| = 0.000000e+00  max tolerance ratio = 0.000e+00
PASS  excited defaults: independent mapping for Ay                      max |Δ| = 0.000000e+00  max tolerance ratio = 0.000e+00
PASS  synthetic mixed signs: djt_hyperfine_tensors return len

## 2. Compare Hamiltonian operators and isolate every component

Complete Hamiltonians are compared at several mixed parameter points. In
addition, every interaction is compared as an isolated matrix difference. This
is important because default relative tolerance on a matrix containing a
hundreds-of-GHz spin-orbit term can otherwise permit MHz-scale differences in
small diagonal terms.

In [5]:
full_hamiltonian_cases = (
    {
        "label": "zero field and zero strain",
        "bx": 0.0,
        "by": 0.0,
        "bz": 0.0,
        "alpha": 0.0,
        "beta": 0.0,
        "upsilon": 0.0,
    },
    {
        "label": "mixed field, strain, and positive IOC",
        "bx": 0.125,
        "by": -0.087,
        "bz": 0.431,
        "alpha": 2.3,
        "beta": -1.7,
        "upsilon": 0.004,
    },
    {
        "label": "custom non-hyperfine parameters",
        "bx": -0.93,
        "by": 0.52,
        "bz": -0.28,
        "alpha": -3.5,
        "beta": 4.2,
        "upsilon": -0.003,
        "q": 0.123,
        "L": 777.7,
        "delta_f": 0.019,
        "rg": 1.1 * params.rg,
    },
)

ZERO_SCALARS = {
    "Aperp": 0.0,
    "Apar": 0.0,
    "A1": 0.0,
    "A2": 0.0,
}
ZERO_TENSORS = independent_djt_tensors(**ZERO_SCALARS)

BASE_NONHYPERFINE = {
    "bx": 0.0,
    "by": 0.0,
    "bz": 0.0,
    "alpha": 0.0,
    "beta": 0.0,
    "rg": 0.0,
    "q": 0.0,
    "L": 0.0,
    "upsilon": 0.0,
    "delta_f": 0.0,
}

# Each entry is (label, baseline overrides, active overrides).
component_specs = (
    ("spin-orbit", {}, {"L": 37.2}),
    ("x strain", {}, {"alpha": 1.23}),
    ("y strain", {}, {"beta": -0.91}),
    ("electron Zeeman x", {}, {"bx": 0.73}),
    ("electron Zeeman y", {}, {"by": -0.61}),
    ("electron Zeeman z", {}, {"bz": 0.57}),
    (
        "nuclear Zeeman x",
        {"bx": 0.73},
        {"bx": 0.73, "rg": 1.7 * params.rg},
    ),
    (
        "nuclear Zeeman y",
        {"by": -0.61},
        {"by": -0.61, "rg": 1.7 * params.rg},
    ),
    (
        "nuclear Zeeman z",
        {"bz": 0.57},
        {"bz": 0.57, "rg": 1.7 * params.rg},
    ),
    (
        "orbital Zeeman z",
        {"bz": 0.57},
        {"bz": 0.57, "q": 0.223},
    ),
    (
        "asymmetric-Ham Zeeman correction",
        {"bz": 0.57},
        {"bz": 0.57, "delta_f": -0.137},
    ),
    ("iso-orbital coupling", {}, {"upsilon": 0.013}),
)

hyperfine_component_specs = (
    ("Aperp hyperfine", {"Aperp": 0.371}),
    ("Apar hyperfine", {"Apar": -0.219}),
    ("A1 hyperfine", {"A1": 0.013}),
    ("A2 hyperfine", {"A2": -0.027}),
)

for manifold in ("ground", "excited"):
    H_original, Href_original, p_original, J2_original = (
        original.create_hamiltonian_nuclear(manifold)
    )
    H_new, Href_new, p_new, J2_new = new.create_hamiltonian_nuclear(
        manifold
    )

    require_length(f"{manifold}: original dipole count", p_original, 3)
    require_length(f"{manifold}: new dipole count", p_new, 3)

    scalar_values = manifold_scalars(manifold)
    A, Ax, Ay = manifold_tensors(manifold)

    # Combined hyperfine check with every non-hyperfine contribution disabled.
    hyperfine_original = H_original(
        **BASE_NONHYPERFINE,
        **scalar_values,
    )
    hyperfine_new = H_new(
        **BASE_NONHYPERFINE,
        A=A,
        Ax=Ax,
        Ay=Ay,
    )
    record_allclose(
        f"{manifold}: isolated combined hyperfine Hamiltonian",
        qutip_dense(hyperfine_original),
        jaxquantum_dense(hyperfine_new),
    )

    record_allclose(
        f"{manifold}: J2",
        qutip_dense(J2_original),
        jaxquantum_dense(J2_new),
    )

    for dipole_index, (old_dipole, new_dipole) in enumerate(
        zip(p_original, p_new, strict=True)
    ):
        record_allclose(
            f"{manifold}: dipole p[{dipole_index}]",
            qutip_dense(old_dipole),
            jaxquantum_dense(new_dipole),
        )

    for case in full_hamiltonian_cases:
        case_kwargs = {
            key: value
            for key, value in case.items()
            if key != "label"
        }

        old_matrix = H_original(**case_kwargs, **scalar_values)
        new_matrix = H_new(**case_kwargs, A=A, Ax=Ax, Ay=Ay)
        record_allclose(
            f"{manifold}: full H -- {case['label']}",
            qutip_dense(old_matrix),
            jaxquantum_dense(new_matrix),
        )

        reference_kwargs = {
            "alpha": case_kwargs["alpha"],
            "beta": case_kwargs["beta"],
        }
        if "L" in case_kwargs:
            reference_kwargs["L"] = case_kwargs["L"]

        record_allclose(
            f"{manifold}: Href -- {case['label']}",
            qutip_dense(Href_original(**reference_kwargs)),
            jaxquantum_dense(Href_new(**reference_kwargs)),
        )

    def original_nonhyperfine_matrix(overrides):
        kwargs = dict(BASE_NONHYPERFINE)
        kwargs.update(overrides)
        kwargs.update(ZERO_SCALARS)
        return qutip_dense(H_original(**kwargs))

    def new_nonhyperfine_matrix(overrides):
        kwargs = dict(BASE_NONHYPERFINE)
        kwargs.update(overrides)
        return jaxquantum_dense(
            H_new(
                **kwargs,
                A=ZERO_TENSORS[0],
                Ax=ZERO_TENSORS[1],
                Ay=ZERO_TENSORS[2],
            )
        )

    for component_name, baseline_overrides, active_overrides in component_specs:
        expected_component = (
            original_nonhyperfine_matrix(active_overrides)
            - original_nonhyperfine_matrix(baseline_overrides)
        )
        actual_component = (
            new_nonhyperfine_matrix(active_overrides)
            - new_nonhyperfine_matrix(baseline_overrides)
        )
        full_name = f"{manifold}: isolated {component_name}"
        require_nonzero_component(full_name, expected_component)
        record_allclose(full_name, expected_component, actual_component)

    for component_name, active_hyperfine in hyperfine_component_specs:
        active_scalars = dict(ZERO_SCALARS)
        active_scalars.update(active_hyperfine)
        active_tensors = independent_djt_tensors(**active_scalars)

        expected_component = qutip_dense(
            H_original(
                **BASE_NONHYPERFINE,
                **active_scalars,
            )
        )
        actual_component = jaxquantum_dense(
            H_new(
                **BASE_NONHYPERFINE,
                A=active_tensors[0],
                Ax=active_tensors[1],
                Ay=active_tensors[2],
            )
        )
        full_name = f"{manifold}: isolated {component_name}"
        require_nonzero_component(full_name, expected_component)
        record_allclose(full_name, expected_component, actual_component)

PASS  ground: original dipole count -- expected 3, received 3
PASS  ground: new dipole count -- expected 3, received 3
PASS  ground: isolated combined hyperfine Hamiltonian                   max |Δ| = 0.000000e+00  max tolerance ratio = 0.000e+00
PASS  ground: J2                                                        max |Δ| = 0.000000e+00  max tolerance ratio = 0.000e+00
PASS  ground: dipole p[0]                                               max |Δ| = 0.000000e+00  max tolerance ratio = 0.000e+00
PASS  ground: dipole p[1]                                               max |Δ| = 0.000000e+00  max tolerance ratio = 0.000e+00
PASS  ground: dipole p[2]                                               max |Δ| = 0.000000e+00  max tolerance ratio = 0.000e+00
PASS  ground: full H -- zero field and zero strain                      max |Δ| = 0.000000e+00  max tolerance ratio = 0.000e+00
PASS  ground: Href -- zero field and zero strain                        max |Δ| = 0.000000e+00  max tolerance rat

## 3. Compare every `solve_hamiltonian` return value

The public function returns six values. This section requires that exact return
length and compares the nested state objects and returned dipoles in addition to
the array-valued eigensystem outputs. A generic sweep and a scalar-field case
exercise both input paths.

In [6]:
B_VALUES = np.array([0.17, 0.63, 1.11])
THETA = 0.71
PHI = -0.43
ALPHA = 2.1
BETA = -1.3
UPSILON = 0.002

solver_cases = (
    {
        "label": "generic three-field sweep",
        "B": B_VALUES,
        "theta": THETA,
        "phi": PHI,
        "alpha": ALPHA,
        "beta": BETA,
        "nonhyperfine": {"upsilon": UPSILON},
        "hyperfine": None,
    },
    {
        "label": "scalar field with custom parameters",
        "B": 0.47,
        "theta": 1.13,
        "phi": 0.82,
        "alpha": -0.62,
        "beta": 0.38,
        "nonhyperfine": {
            "rg": 1.4 * params.rg,
            "q": 0.187,
            "L": 521.3,
            "upsilon": -0.006,
            "delta_f": 0.041,
        },
        "hyperfine": {
            "Aperp": 0.271,
            "Apar": -0.113,
            "A1": 0.009,
            "A2": -0.021,
        },
    },
)

for case in solver_cases:
    for manifold in ("ground", "excited"):
        hyperfine_scalars = (
            manifold_scalars(manifold)
            if case["hyperfine"] is None
            else dict(case["hyperfine"])
        )
        A, Ax, Ay = independent_djt_tensors(**hyperfine_scalars)

        old_kwargs = dict(case["nonhyperfine"])
        old_kwargs.update(hyperfine_scalars)
        new_kwargs = dict(case["nonhyperfine"])
        new_kwargs.update({"A": A, "Ax": Ax, "Ay": Ay})

        old_solution = original.solve_hamiltonian(
            case["B"],
            case["theta"],
            case["phi"],
            manifold=manifold,
            alpha=case["alpha"],
            beta=case["beta"],
            **old_kwargs,
        )
        new_solution = new.solve_hamiltonian(
            case["B"],
            case["theta"],
            case["phi"],
            manifold=manifold,
            alpha=case["alpha"],
            beta=case["beta"],
            **new_kwargs,
        )

        prefix = f"{manifold}: solve_hamiltonian -- {case['label']}"
        require_length(f"{prefix}: original return length", old_solution, 6)
        require_length(f"{prefix}: new return length", new_solution, 6)

        old_state_columns = qutip_state_columns(old_solution[3])
        new_state_columns = jaxquantum_state_columns(new_solution[3])

        record_allclose(
            f"{prefix}: eigenvalues",
            old_solution[0],
            new_solution[0],
        )
        record_allclose(
            f"{prefix}: reference eigenvalues",
            old_solution[1],
            new_solution[1],
        )
        record_allclose(
            f"{prefix}: adjacent eigenvalue gaps",
            jnp.diff(old_solution[0], axis=-1),
            jnp.diff(new_solution[0], axis=-1),
        )
        record_allclose(
            f"{prefix}: adjacent reference-energy gaps",
            jnp.diff(old_solution[1], axis=-1),
            jnp.diff(new_solution[1], axis=-1),
        )
        record_allclose(
            f"{prefix}: eigenvector-matrix projectors",
            state_projectors(old_solution[2]),
            state_projectors(new_solution[2]),
        )
        record_allclose(
            f"{prefix}: nested-state projectors",
            state_projectors(old_state_columns),
            state_projectors(new_state_columns),
        )
        record_allclose(
            f"{prefix}: original U agrees with original U_states",
            old_solution[2],
            old_state_columns,
        )
        record_allclose(
            f"{prefix}: new U agrees with new U_states",
            new_solution[2],
            new_state_columns,
        )
        record_allclose(
            f"{prefix}: J2 alignment",
            old_solution[4],
            new_solution[4],
        )

        require_length(f"{prefix}: original returned dipole count", old_solution[5], 3)
        require_length(f"{prefix}: new returned dipole count", new_solution[5], 3)
        for dipole_index, (old_dipole, new_dipole) in enumerate(
            zip(old_solution[5], new_solution[5], strict=True)
        ):
            record_allclose(
                f"{prefix}: returned dipole p[{dipole_index}]",
                qutip_dense(old_dipole),
                jaxquantum_dense(new_dipole),
            )

PASS  ground: solve_hamiltonian -- generic three-field sweep: original return length -- expected 6, received 6
PASS  ground: solve_hamiltonian -- generic three-field sweep: new return length -- expected 6, received 6
PASS  ground: solve_hamiltonian -- generic three-field sweep: eigenvalues  max |Δ| = 1.534772e-12  max tolerance ratio = 3.700e-10
PASS  ground: solve_hamiltonian -- generic three-field sweep: reference eigenvalues  max |Δ| = 2.273737e-13  max tolerance ratio = 5.479e-11
PASS  ground: solve_hamiltonian -- generic three-field sweep: adjacent eigenvalue gaps  max |Δ| = 2.785328e-12  max tolerance ratio = 2.064e-06
PASS  ground: solve_hamiltonian -- generic three-field sweep: adjacent reference-energy gaps  max |Δ| = 1.705303e-13  max tolerance ratio = 1.705e-05
PASS  ground: solve_hamiltonian -- generic three-field sweep: eigenvector-matrix projectors  max |Δ| = 5.016978e-12  max tolerance ratio = 2.229e-05
PASS  ground: solve_hamiltonian -- generic three-field sweep: nested

## 4. Compare cyclicity and optical functions

The optical tests use nonzero field, strain, IOC, and a complex polarization
vector with all three components present. Tuple lengths are checked before any
positional indexing.

In [7]:
example_rates = jnp.array(
    [
        [1.0, 2.0, 3.0, 4.0],
        [0.0, 0.0, 0.0, 0.0],
        [5.0, 1.0, 0.5, 2.5],
    ],
    dtype=jnp.float64,
)
expected_cyclicity = jnp.array(
    [
        [0.1, 0.2, 0.3, 0.4],
        [0.0, 0.0, 0.0, 0.0],
        [5.0 / 9.0, 1.0 / 9.0, 0.5 / 9.0, 2.5 / 9.0],
    ],
    dtype=jnp.float64,
)
old_example_cyclicity = original.calculate_cyclicity(
    np.asarray(example_rates)
)
new_example_cyclicity = new.calculate_cyclicity(example_rates)
record_allclose(
    "calculate_cyclicity: original versus analytic result",
    expected_cyclicity,
    old_example_cyclicity,
)
record_allclose(
    "calculate_cyclicity: new versus analytic result",
    expected_cyclicity,
    new_example_cyclicity,
)
record_allclose(
    "calculate_cyclicity: original versus new",
    old_example_cyclicity,
    new_example_cyclicity,
)

A_gnd, Ax_gnd, Ay_gnd = manifold_tensors("ground")
A_exc, Ax_exc, Ay_exc = manifold_tensors("excited")

old_gnd_kwargs = {"upsilon": 0.0015}
old_exc_kwargs = {"upsilon": -0.0007}
new_gnd_kwargs = {
    "upsilon": 0.0015,
    "A": A_gnd,
    "Ax": Ax_gnd,
    "Ay": Ay_gnd,
}
new_exc_kwargs = {
    "upsilon": -0.0007,
    "A": A_exc,
    "Ax": Ax_exc,
    "Ay": Ay_exc,
}

ETA = np.array([0.31, -0.44, 0.77], dtype=np.complex128)
ALPHA_EXC = -1.8
BETA_EXC = 0.9

old_ple = original.PLE_transitions(
    B_VALUES,
    THETA,
    PHI,
    ETA,
    alpha=ALPHA,
    beta=BETA,
    alpha_exc=ALPHA_EXC,
    beta_exc=BETA_EXC,
    gnd_kwargs=old_gnd_kwargs,
    exc_kwargs=old_exc_kwargs,
)
new_ple = new.PLE_transitions(
    B_VALUES,
    THETA,
    PHI,
    ETA,
    alpha=ALPHA,
    beta=BETA,
    alpha_exc=ALPHA_EXC,
    beta_exc=BETA_EXC,
    gnd_kwargs=new_gnd_kwargs,
    exc_kwargs=new_exc_kwargs,
)
require_length("PLE_transitions: original return length", old_ple, 10)
require_length("PLE_transitions: new return length", new_ple, 10)

for index, label in (
    (0, "ground energies"),
    (1, "ground reference energies"),
    (3, "ground alignment"),
    (4, "excited energies"),
    (5, "excited reference energies"),
    (7, "excited alignment"),
    (8, "transition intensities"),
    (9, "emission branching ratios"),
):
    record_allclose(
        f"PLE_transitions: {label}",
        old_ple[index],
        new_ple[index],
    )

record_allclose(
    "PLE_transitions: ground eigenstate projectors",
    state_projectors(old_ple[2]),
    state_projectors(new_ple[2]),
)
record_allclose(
    "PLE_transitions: excited eigenstate projectors",
    state_projectors(old_ple[6]),
    state_projectors(new_ple[6]),
)
record_allclose(
    "PLE_transitions: original branching rows sum to one",
    jnp.ones(old_ple[9].shape[:-1]),
    jnp.sum(old_ple[9], axis=-1),
)
record_allclose(
    "PLE_transitions: new branching rows sum to one",
    jnp.ones(new_ple[9].shape[:-1]),
    jnp.sum(new_ple[9], axis=-1),
)

PASS  calculate_cyclicity: original versus analytic result              max |Δ| = 0.000000e+00  max tolerance ratio = 0.000e+00
PASS  calculate_cyclicity: new versus analytic result                   max |Δ| = 0.000000e+00  max tolerance ratio = 0.000e+00
PASS  calculate_cyclicity: original versus new                          max |Δ| = 0.000000e+00  max tolerance ratio = 0.000e+00
PASS  PLE_transitions: original return length -- expected 10, received 10
PASS  PLE_transitions: new return length -- expected 10, received 10
PASS  PLE_transitions: ground energies                                  max |Δ| = 1.932676e-12  max tolerance ratio = 4.659e-10
PASS  PLE_transitions: ground reference energies                        max |Δ| = 2.273737e-13  max tolerance ratio = 5.479e-11
PASS  PLE_transitions: ground alignment                                 max |Δ| = 2.700062e-13  max tolerance ratio = 2.630e-08
PASS  PLE_transitions: excited energies                                 max |Δ| = 3.86535

True

### Folded optical-cycling cyclicity

In addition to comparing all seven outputs, this section checks the defining
population-conservation identity: each row of `emission_folded` must sum to the
total direct emission rate.

In [8]:
old_spinflip = original.calculate_cyclicity_spinflip(
    B_VALUES,
    THETA,
    PHI,
    alpha=ALPHA,
    beta=BETA,
    alpha_exc=ALPHA_EXC,
    beta_exc=BETA_EXC,
    gnd_kwargs=old_gnd_kwargs,
    exc_kwargs=old_exc_kwargs,
    cap=1e6,
)
new_spinflip = new.calculate_cyclicity_spinflip(
    B_VALUES,
    THETA,
    PHI,
    alpha=ALPHA,
    beta=BETA,
    alpha_exc=ALPHA_EXC,
    beta_exc=BETA_EXC,
    gnd_kwargs=new_gnd_kwargs,
    exc_kwargs=new_exc_kwargs,
    cap=1e6,
)
require_length(
    "calculate_cyclicity_spinflip: original return length",
    old_spinflip,
    7,
)
require_length(
    "calculate_cyclicity_spinflip: new return length",
    new_spinflip,
    7,
)

for index, label in enumerate(
    (
        "ground energies",
        "excited energies",
        "spontaneous-emission rates",
        "folded per-line cyclicity",
        "ground spin signs",
        "excited spin signs",
        "folded emission rates",
    )
):
    record_allclose(
        f"calculate_cyclicity_spinflip: {label}",
        old_spinflip[index],
        new_spinflip[index],
    )

record_allclose(
    "calculate_cyclicity_spinflip: original folded-rate conservation",
    jnp.sum(old_spinflip[2], axis=-1),
    jnp.sum(old_spinflip[6], axis=-1),
)
record_allclose(
    "calculate_cyclicity_spinflip: new folded-rate conservation",
    jnp.sum(new_spinflip[2], axis=-1),
    jnp.sum(new_spinflip[6], axis=-1),
)
num_lower = old_spinflip[3].shape[-1] // 2
record_allclose(
    "calculate_cyclicity_spinflip: original upper-ground cyclicity is zero",
    jnp.zeros_like(old_spinflip[3][..., num_lower:]),
    old_spinflip[3][..., num_lower:],
)
record_allclose(
    "calculate_cyclicity_spinflip: new upper-ground cyclicity is zero",
    jnp.zeros_like(new_spinflip[3][..., num_lower:]),
    new_spinflip[3][..., num_lower:],
)

PASS  calculate_cyclicity_spinflip: original return length -- expected 7, received 7
PASS  calculate_cyclicity_spinflip: new return length -- expected 7, received 7
PASS  calculate_cyclicity_spinflip: ground energies                     max |Δ| = 1.932676e-12  max tolerance ratio = 4.659e-10
PASS  calculate_cyclicity_spinflip: excited energies                    max |Δ| = 3.865352e-12  max tolerance ratio = 2.576e-10
PASS  calculate_cyclicity_spinflip: spontaneous-emission rates          max |Δ| = 3.377298e-12  max tolerance ratio = 2.809e-04
PASS  calculate_cyclicity_spinflip: folded per-line cyclicity           max |Δ| = 1.137849e-05  max tolerance ratio = 5.420e-04
PASS  calculate_cyclicity_spinflip: ground spin signs                   max |Δ| = 0.000000e+00  max tolerance ratio = 0.000e+00
PASS  calculate_cyclicity_spinflip: excited spin signs                  max |Δ| = 0.000000e+00  max tolerance ratio = 0.000e+00
PASS  calculate_cyclicity_spinflip: folded emission rates          

True

### Direct checks of `_folded_cyclicity_weight`

Testing each control independently prevents a correct combined spectrum from
hiding an error in one unused or weakly contributing branch.

In [9]:
example_folded_cyclicity = np.array(
    [
        [
            [0.2, 0.8, 0.0, 0.0],
            [1.5, 3.0, 0.0, 0.0],
            [0.05, 7.0, 0.0, 0.0],
            [11.0, 0.4, 0.0, 0.0],
        ],
        [
            [0.6, 2.0, 0.0, 0.0],
            [4.0, 0.1, 0.0, 0.0],
            [0.3, 0.9, 0.0, 0.0],
            [8.0, 1.2, 0.0, 0.0],
        ],
    ],
    dtype=float,
)

weight_cases = (
    ("no controls", {}),
    (
        "smooth weighting only",
        {"cyclicity_weight": True, "cyclicity_half": 1.7},
    ),
    (
        "minimum gate only",
        {
            "cyclicity_min": 0.75,
            "cyclicity_softness": 3.0,
        },
    ),
    (
        "combined weighting and minimum gate",
        {
            "cyclicity_min": 0.75,
            "cyclicity_weight": True,
            "cyclicity_half": 1.7,
            "cyclicity_softness": 3.0,
        },
    ),
)

for label, kwargs in weight_cases:
    old_weight = original._folded_cyclicity_weight(
        example_folded_cyclicity,
        **kwargs,
    )
    new_weight = new._folded_cyclicity_weight(
        example_folded_cyclicity,
        **kwargs,
    )
    require_length(
        f"_folded_cyclicity_weight -- {label}: original return length",
        old_weight,
        2,
    )
    require_length(
        f"_folded_cyclicity_weight -- {label}: new return length",
        new_weight,
        2,
    )
    record_allclose(
        f"_folded_cyclicity_weight -- {label}: full weights",
        old_weight[0],
        new_weight[0],
    )
    record_allclose(
        f"_folded_cyclicity_weight -- {label}: lower cyclicity",
        old_weight[1],
        new_weight[1],
    )

PASS  _folded_cyclicity_weight -- no controls: original return length -- expected 2, received 2
PASS  _folded_cyclicity_weight -- no controls: new return length -- expected 2, received 2
PASS  _folded_cyclicity_weight -- no controls: full weights             max |Δ| = 0.000000e+00  max tolerance ratio = 0.000e+00
PASS  _folded_cyclicity_weight -- no controls: lower cyclicity          max |Δ| = 0.000000e+00  max tolerance ratio = 0.000e+00
PASS  _folded_cyclicity_weight -- smooth weighting only: original return length -- expected 2, received 2
PASS  _folded_cyclicity_weight -- smooth weighting only: new return length -- expected 2, received 2
PASS  _folded_cyclicity_weight -- smooth weighting only: full weights   max |Δ| = 0.000000e+00  max tolerance ratio = 0.000e+00
PASS  _folded_cyclicity_weight -- smooth weighting only: lower cyclicity  max |Δ| = 0.000000e+00  max tolerance ratio = 0.000e+00
PASS  _folded_cyclicity_weight -- minimum gate only: original return length -- expected 2, r

### Bare, weighted, and scalar-input PLE spectra

The frequency samples include every QuTiP transition center and half-linewidth
offsets, avoiding a weak test in which both spectra are almost zero over an
unrelated frequency window.

In [10]:
LINEWIDTH = 0.09
transition_centers = (
    (old_ple[4] - old_ple[5][0])[:, :, None]
    - (old_ple[0] - old_ple[1][0])[:, None, :]
)
transition_centers = np.asarray(transition_centers).reshape(-1)
FREQUENCIES = np.unique(
    np.concatenate(
        (
            transition_centers,
            transition_centers - LINEWIDTH / 2.0,
            transition_centers + LINEWIDTH / 2.0,
            np.linspace(
                float(np.min(transition_centers)) - LINEWIDTH,
                float(np.max(transition_centers)) + LINEWIDTH,
                401,
            ),
        )
    )
)

common_spectrum_kwargs = dict(
    f_meas=FREQUENCIES,
    B=B_VALUES,
    theta=THETA,
    phi=PHI,
    eta=ETA,
    intensity=0.7,
    lw=LINEWIDTH,
    alpha=ALPHA,
    beta=BETA,
    alpha_exc=ALPHA_EXC,
    beta_exc=BETA_EXC,
)

old_bare_spectrum = original.PLE_spectrum(
    **common_spectrum_kwargs,
    gnd_kwargs=old_gnd_kwargs,
    exc_kwargs=old_exc_kwargs,
)
new_bare_spectrum = new.PLE_spectrum(
    **common_spectrum_kwargs,
    gnd_kwargs=new_gnd_kwargs,
    exc_kwargs=new_exc_kwargs,
)
record_allclose(
    "PLE_spectrum: bare spectrum",
    old_bare_spectrum,
    new_bare_spectrum,
)

old_weighted_spectrum, old_returned_cyclicity = original.PLE_spectrum(
    **common_spectrum_kwargs,
    gnd_kwargs=old_gnd_kwargs,
    exc_kwargs=old_exc_kwargs,
    cyclicity_min=0.5,
    cyclicity_weight=True,
    cyclicity_half=1.0,
    cyclicity_softness=4.0,
    cyclicity_cap=1e6,
    return_cyclicity=True,
)
new_weighted_spectrum, new_returned_cyclicity = new.PLE_spectrum(
    **common_spectrum_kwargs,
    gnd_kwargs=new_gnd_kwargs,
    exc_kwargs=new_exc_kwargs,
    cyclicity_min=0.5,
    cyclicity_weight=True,
    cyclicity_half=1.0,
    cyclicity_softness=4.0,
    cyclicity_cap=1e6,
    return_cyclicity=True,
)
record_allclose(
    "PLE_spectrum: folded-cyclicity-weighted spectrum",
    old_weighted_spectrum,
    new_weighted_spectrum,
)
record_allclose(
    "PLE_spectrum: returned folded cyclicity",
    old_returned_cyclicity,
    new_returned_cyclicity,
)

# Exercise scalar-B squeezing and return_cyclicity without applying a brightness gate.
scalar_spectrum_kwargs = dict(common_spectrum_kwargs)
scalar_spectrum_kwargs["B"] = float(B_VALUES[1])
old_scalar_spectrum = original.PLE_spectrum(
    **scalar_spectrum_kwargs,
    gnd_kwargs=old_gnd_kwargs,
    exc_kwargs=old_exc_kwargs,
    return_cyclicity=True,
)
new_scalar_spectrum = new.PLE_spectrum(
    **scalar_spectrum_kwargs,
    gnd_kwargs=new_gnd_kwargs,
    exc_kwargs=new_exc_kwargs,
    return_cyclicity=True,
)
require_length(
    "PLE_spectrum scalar input: original return length",
    old_scalar_spectrum,
    2,
)
require_length(
    "PLE_spectrum scalar input: new return length",
    new_scalar_spectrum,
    2,
)
record_structure(
    "PLE_spectrum scalar input: original spectrum is one-dimensional",
    np.ndim(old_scalar_spectrum[0]) == 1,
    f"shape={np.shape(old_scalar_spectrum[0])}",
)
record_structure(
    "PLE_spectrum scalar input: new spectrum is one-dimensional",
    np.ndim(new_scalar_spectrum[0]) == 1,
    f"shape={np.shape(new_scalar_spectrum[0])}",
)
record_allclose(
    "PLE_spectrum: scalar-input spectrum",
    old_scalar_spectrum[0],
    new_scalar_spectrum[0],
)
record_allclose(
    "PLE_spectrum: scalar-input returned cyclicity",
    old_scalar_spectrum[1],
    new_scalar_spectrum[1],
)

PASS  PLE_spectrum: bare spectrum                                       max |Δ| = 1.078240e-10  max tolerance ratio = 8.106e-06
PASS  PLE_spectrum: folded-cyclicity-weighted spectrum                  max |Δ| = 1.078080e-10  max tolerance ratio = 8.039e-06
PASS  PLE_spectrum: returned folded cyclicity                           max |Δ| = 1.137849e-05  max tolerance ratio = 5.420e-04
PASS  PLE_spectrum scalar input: original return length -- expected 2, received 2
PASS  PLE_spectrum scalar input: new return length -- expected 2, received 2
PASS  PLE_spectrum scalar input: original spectrum is one-dimensional -- shape=(977,)
PASS  PLE_spectrum scalar input: new spectrum is one-dimensional -- shape=(977,)
PASS  PLE_spectrum: scalar-input spectrum                               max |Δ| = 7.294609e-11  max tolerance ratio = 6.662e-06
PASS  PLE_spectrum: scalar-input returned cyclicity                     max |Δ| = 1.138044e-05  max tolerance ratio = 5.421e-04


True

## 5. Final result

The final cell verifies the expected number of numerical comparisons, rejects
duplicate check names, prints all results, and raises if any numerical or
structural check failed.

In [11]:
EXPECTED_NUMERIC_CHECKS = 150

record_structure(
    "expected numerical check count",
    len(numeric_results) == EXPECTED_NUMERIC_CHECKS,
    f"expected {EXPECTED_NUMERIC_CHECKS}, received {len(numeric_results)}",
)
numeric_names = [result["name"] for result in numeric_results]
record_structure(
    "numerical check names are unique",
    len(numeric_names) == len(set(numeric_names)),
)
structure_names = [result["name"] for result in structure_results]
record_structure(
    "structure check names are unique",
    len(structure_names) == len(set(structure_names)),
)

print(
    f"\n{'Result':6s}  {'Maximum |difference|':>22s}  "
    f"{'Max tolerance ratio':>19s}  Check"
)
print("-" * 130)
for result in numeric_results:
    status = "PASS" if result["passed"] else "FAIL"
    print(
        f"{status:6s}  "
        f"{result['max_abs_difference']:22.6e}  "
        f"{result['max_tolerance_ratio']:19.6e}  "
        f"{result['name']}"
    )

numeric_failures = [
    result for result in numeric_results if not result["passed"]
]
structure_failures = [
    result for result in structure_results if not result["passed"]
]

print("-" * 130)
print(
    f"Passed {len(numeric_results) - len(numeric_failures)} of "
    f"{len(numeric_results)} numerical checks and "
    f"{len(structure_results) - len(structure_failures)} of "
    f"{len(structure_results)} structure checks."
)

assert not numeric_failures, (
    "Numerical validation failed for: "
    + ", ".join(result["name"] for result in numeric_failures)
)
assert not structure_failures, (
    "Structural validation failed for: "
    + ", ".join(result["name"] for result in structure_failures)
)

print("ALL AUDITED VALIDATION CHECKS PASSED")

PASS  expected numerical check count -- expected 150, received 150
PASS  numerical check names are unique
PASS  structure check names are unique

Result    Maximum |difference|  Max tolerance ratio  Check
----------------------------------------------------------------------------------------------------------------------------------
PASS              0.000000e+00         0.000000e+00  ground defaults: independent mapping for A
PASS              0.000000e+00         0.000000e+00  ground defaults: independent mapping for Ax
PASS              0.000000e+00         0.000000e+00  ground defaults: independent mapping for Ay
PASS              0.000000e+00         0.000000e+00  excited defaults: independent mapping for A
PASS              0.000000e+00         0.000000e+00  excited defaults: independent mapping for Ax
PASS              0.000000e+00         0.000000e+00  excited defaults: independent mapping for Ay
PASS              0.000000e+00         0.000000e+00  synthetic mixed signs: indep